# 1. Setup

Combine the frozen 3Bv2 and 3Cv1 candidate probabilities. Inference-only experiment inspired by [Weighted ensemble study](https://www.sciencedirect.com/science/article/pii/S2590005626006715). Set the run folder(s) to the local checkpoint directories containing selection.json and the .pt weights. For the ensemble, the publisher page was unavailable when preparing this notebook; no paper metrics are assumed.

In [ ]:
import json, math, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from timm.layers import set_reentrant_ckpt
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_crop_and_disease_names, load_split, get_final_test_loaders
from common.evaluate import evaluate, val_accuracy, concept_accuracy
MODEL_NAME = 'approach3bc_dino_weighted_ensemble'
B_RUN = None  # Completed 3Bv2 checkpoint run folder.
C_RUN = None  # Completed 3Cv1 checkpoint run folder.
WEIGHTS = (0.5, 0.5)  # 3Bv2, 3Cv1
cfg.NUM_WORKERS = 0
cfg.set_seed(cfg.SEED)

# 2. Data

Development uses the common training-derived candidate order.

In [ ]:
pairs = get_pairs()
concept_names = get_crop_and_disease_names()

# 3. Model

Restore the original trained architectures and combine independently normalized candidate scores.

In [ ]:
MODEL = 'vit_base_patch14_dinov2.lvd142m'

def c_components(HP):
    def snapshot(repository,revision,allow_download,patterns):
        from huggingface_hub import snapshot_download
        path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
        return path

    class Dino(nn.Module):
        """Exact two-independent-tower initialization from ToTnki's released runner."""
        def __init__(self,config,device,allow_download=False,revision=None):
            super().__init__()
            import timm
            path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                          ['config.json','model.safetensors','pytorch_model.bin'])
            weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
            def pinned_source_model(classes):
                # Same source factory, with only an explicit weight-file binding added.
                model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                        pretrained_cfg_overlay={'file':str(weights)})
                model.requires_grad_(False);model.head.requires_grad_(True)
                return model
            self.crop=pinned_source_model(cfg.NUM_CROPS)
            self.disease=pinned_source_model(cfg.NUM_DISEASES)
            self.augmentation_rng_state=torch.get_rng_state().clone()
            self.width=self.crop.num_features
            self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
                architecture=MODEL,width=self.width,independent_visual_towers=2,
                initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
                preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
            self.to(device)
        def policy(self, depth):
            self.requires_grad_(False)
            for tower in (self.crop, self.disease):
                tower.blocks[-depth:].requires_grad_(True)
                tower.set_grad_checkpointing(True)

        def features(self,pixels):
            return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                    self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
        def forward(self,pixels):return self.features(pixels)

    def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))


    RECIPE = dict(synthetic_pairs='observed_training_only', loss='moving_weighted_sum')

    class Model(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone
            width, hidden, dim = backbone.width, HP['hidden_dim'], HP['feature_dim']
            self.crop_projection = projection(width, hidden, dim)
            self.disease_projection = projection(width, hidden, dim)
            self.crop_aux = nn.Linear(dim, cfg.NUM_CROPS)
            self.disease_aux = nn.Linear(dim, cfg.NUM_DISEASES)
            self.attention = nn.MultiheadAttention(dim, HP['fusion_attention_heads'], batch_first=True)
            self.image_fusion = nn.Sequential(nn.LayerNorm(2 * dim), nn.Linear(2 * dim, dim), nn.GELU())
            self.crop_embedding = nn.Embedding(cfg.NUM_CROPS, dim)
            self.disease_embedding = nn.Embedding(cfg.NUM_DISEASES, dim)
            self.pair_fusion = nn.Sequential(nn.Linear(2 * dim, dim), nn.GELU())

        def features(self, images):
            crop, disease = self.backbone(images)
            return self.crop_projection(crop), self.disease_projection(disease)

        def score(self, crop, disease, candidate_pairs):
            combined = crop + disease
            tokens = torch.stack((crop + combined, disease + combined), dim=1)
            attended, _ = self.attention(tokens, tokens, tokens, need_weights=False)
            residuals = torch.stack((disease, crop), dim=1)
            image = self.image_fusion((attended + residuals).flatten(1))
            crop_ids, disease_ids = candidate_pairs.unbind(1)
            candidates = self.pair_fusion(torch.cat(
                (self.crop_embedding(crop_ids), self.disease_embedding(disease_ids)), dim=1))
            return image @ candidates.T


    def make_model(depth, metadata=None):
        cfg.set_seed(cfg.SEED)
        metadata = metadata or {}
        backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
        backbone.policy(depth)
        cfg.set_seed(cfg.SEED)
        model = Model(backbone).to(cfg.DEVICE)
        torch.set_rng_state(backbone.augmentation_rng_state)
        return model, dict(dino=backbone.metadata)


    def make_predict(model, candidate_pairs):
        candidates = candidate_pairs.to(cfg.DEVICE)

        @torch.no_grad()
        def predict(images):
            model.eval()
            return model.score(*model.features(images.to(cfg.DEVICE)), candidates)
        return predict
    return make_model, RECIPE


def b_components(HP):
    def snapshot(repository,revision,allow_download,patterns):
        from huggingface_hub import snapshot_download
        path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
        return path

    class Dino(nn.Module):
        """Exact two-independent-tower initialization from ToTnki's released runner."""
        def __init__(self,config,device,allow_download=False,revision=None):
            super().__init__()
            import timm
            path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                          ['config.json','model.safetensors','pytorch_model.bin'])
            weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
            def pinned_source_model(classes):
                # Same source factory, with only an explicit weight-file binding added.
                model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                        pretrained_cfg_overlay={'file':str(weights)})
                model.requires_grad_(False);model.head.requires_grad_(True)
                return model
            self.crop=pinned_source_model(cfg.NUM_CROPS)
            self.disease=pinned_source_model(cfg.NUM_DISEASES)
            self.augmentation_rng_state=torch.get_rng_state().clone()
            self.width=self.crop.num_features
            self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
                architecture=MODEL,width=self.width,independent_visual_towers=2,
                initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
                preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
            self.to(device)
        def policy(self, depth):
            self.requires_grad_(False)
            for tower in (self.crop, self.disease):
                tower.blocks[-depth:].requires_grad_(True)
                tower.set_grad_checkpointing(True)

        def features(self,pixels):
            return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                    self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
        def forward(self,pixels):return self.features(pixels)

    class TextOnly:
        """No CLIPModel, CLIPVisionModel, OpenCLIP model, or image encoder is created."""
        def __init__(self,config,device,allow_download=False,revision=None):
            from transformers import CLIPTextModelWithProjection, CLIPTokenizerFast
            path=snapshot('openai/clip-vit-base-patch16',revision or config['text_revision'],allow_download,
                ['*.json','vocab.json','merges.txt','model.safetensors','pytorch_model.bin'])
            self.model=CLIPTextModelWithProjection.from_pretrained(path,local_files_only=True).to(device).eval().requires_grad_(False)
            self.tokenizer=CLIPTokenizerFast.from_pretrained(path,local_files_only=True)
            if any('vision' in name for name,_ in self.model.named_modules()):raise RuntimeError('A CLIP vision encoder was instantiated.')
            self.dimension=self.model.config.projection_dim;self.device=device
            if self.dimension!=512:raise ValueError('Expected the CLIP B/16 512-dimensional text projection.')
            weight=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
            self.metadata=dict(repository='openai/clip-vit-base-patch16',revision=path.name,
                class_name=type(self.model).__name__,dimension=self.dimension,trainable_parameters=0,clip_image_encoder=False)
        @torch.no_grad()
        def encode(self,prompts,size):
            self.assert_frozen();result=[]
            for start in range(0,len(prompts),size):
                tokens=self.tokenizer(prompts[start:start+size],padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
                result.append(F.normalize(self.model(**tokens).text_embeds.float(),dim=1))
            return torch.cat(result)
        def assert_frozen(self):
            if self.model.training or any(p.requires_grad or p.grad is not None for p in self.model.parameters()):
                raise RuntimeError('CLIP text encoder must be eval, frozen and gradient-free.')

    def text_banks(text,pairs,names,config,ensemble=False):
        crops,diseases=names
        def clean(x):return x.replace('_',' ')
        pair_groups=[]
        for c,d in pairs.tolist():
            if clean(diseases[d]).strip().casefold()=='healthy':
                templates=config['healthy_templates'] if ensemble else config['healthy_templates'][:1]
            else:templates=config['pair_templates'] if ensemble else config['pair_templates'][:1]
            pair_groups.append([t.format(crop=clean(crops[c]),disease=clean(diseases[d])) for t in templates])
        flat=[p for group in pair_groups for p in group];encoded=text.encode(flat,config['text_batch_size'])
        bank=[];offset=0
        for group in pair_groups:
            bank.append(F.normalize(encoded[offset:offset+len(group)].mean(0),dim=0));offset+=len(group)
        crop_prompts=[f'a photo of a {clean(crops[i])} plant' for i in range(cfg.NUM_CROPS)]
        disease_prompts=[('a photo of a healthy leaf' if clean(diseases[i]).casefold()=='healthy' else f'a leaf showing {clean(diseases[i])}') for i in range(cfg.NUM_DISEASES)]
        return [torch.stack(bank),text.encode(crop_prompts,config['text_batch_size']),text.encode(disease_prompts,config['text_batch_size'])],dict(pair=pair_groups,crop=crop_prompts,disease=disease_prompts)

    def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))

    def zscore(x):
        return (x-x.mean(1, keepdim=True)) / x.std(1, unbiased=False, keepdim=True).clamp_min(1e-6)

    def factorized(parts, pairs, weights):
        pair, crop, disease = parts
        if len(weights) != 3 or min(weights) < 0 or abs(sum(weights)-1) > 1e-6:
            raise ValueError('Three nonnegative scorer weights summing to one required.')
        pairs = pairs.to(pair.device)
        return weights[0]*zscore(pair)+weights[1]*zscore(crop)[:, pairs[:,0]]+weights[2]*zscore(disease)[:, pairs[:,1]]


    RECIPE = dict(pair=1., primitive=1.)


    class Model(nn.Module):
        def __init__(self, backbone, banks):
            super().__init__()
            self.backbone = backbone
            width, hidden = backbone.width, HP['hidden_dim']
            dim = banks[0].shape[1]
            self.crop_projection = projection(width, hidden, dim)
            self.disease_projection = projection(width, hidden, dim)

            self.pair_projection = projection(width, hidden, dim)
            self.log_temperature = nn.Parameter(torch.tensor(math.log(1 / .07)))
            for index, bank in enumerate(banks):
                self.register_buffer(f'text{index}', bank)

        def install_text(self, banks):
            for index, bank in enumerate(banks):
                setattr(self, f'text{index}', bank)

        def forward(self, pixels):
            crop_features, disease_features = self.backbone(pixels)
            crop = F.normalize(self.crop_projection(crop_features), dim=1)
            disease = F.normalize(self.disease_projection(disease_features), dim=1)
            pair = F.normalize(self.pair_projection((crop_features + disease_features) / 2), dim=1)
            scale = self.log_temperature.clamp(0., math.log(100.)).exp()
            return scale * pair @ self.text0.T, scale * crop @ self.text1.T, scale * disease @ self.text2.T


    def make_model(pairs, names, depth, metadata=None):
        cfg.set_seed(cfg.SEED)
        metadata = metadata or {}
        backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
        backbone.policy(depth)
        text, banks = None, None

        text = TextOnly(HP, cfg.DEVICE, allow_download=True, revision=(metadata.get('text') or {}).get('revision'))
        banks, _ = text_banks(text, pairs, names, HP)

        torch.manual_seed(cfg.SEED)
        model = Model(backbone, banks).to(cfg.DEVICE)
        torch.set_rng_state(backbone.augmentation_rng_state)
        return model, text, dict(dino=backbone.metadata, text=text.metadata if text else None)


    def make_predict(model, candidate_pairs):
        @torch.no_grad()
        def predict(images):
            model.eval()
            return factorized(model(images.to(cfg.DEVICE)), candidate_pairs, HP['score_weights'])
        return predict
    return make_model, text_banks, make_predict, RECIPE

def load_run(folder, model_name, family, recipe):
    run = Path(folder)
    selection = json.loads((run / 'selection.json').read_text(encoding='utf8'))
    checkpoint = torch.load(run / selection['checkpoint'], map_location='cpu', weights_only=True)
    expected = dict(model_name=model_name, family=family, debug=False, recipe=recipe)
    if any(record.get(key) != value for record in (selection, checkpoint)
           for key, value in expected.items()):
        raise ValueError(f'Expected a full run from {model_name}.')
    if (not torch.equal(checkpoint['pairs'], pairs) or selection['development_pairs'] != pairs.tolist()
            or any(selection[key] != checkpoint[key] for key in ('hp', 'depth', 'best_epoch', 'best_val'))):
        raise ValueError('Checkpoint and selection do not match.')
    hp = checkpoint['hp']
    if hp['backbone'] != MODEL or hp['norm'] != 'imagenet' or hp['seed'] != cfg.SEED:
        raise ValueError('Unexpected backbone, preprocessing or seed.')
    if checkpoint['depth'] not in hp['unfreeze_blocks'] or checkpoint['depth'] not in (2, 4, 6, 8):
        raise ValueError('Checkpoint depth does not match the sweep.')
    train, val, _, _ = get_loaders(hp['batch_size'], norm=hp['norm'], debug=False)
    if (selection['train_images'] != train.dataset.frame.image.tolist()
            or selection['validation_images'] != val.dataset.frame.image.tolist()):
        raise ValueError('Shared train/validation membership changed.')
    return checkpoint


def freeze(model, checkpoint, metadata):
    if metadata != checkpoint['metadata']:
        raise ValueError('Pretrained model identity changed.')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval().requires_grad_(False)
    for tower in (model.backbone.crop, model.backbone.disease):
        tower.set_grad_checkpointing(False)
    return model


def soft_vote(scores, weights):
    return sum(weight * branch.float().softmax(dim=1)
               for weight, branch in zip(weights, scores, strict=True))

# 4. Training

No retraining. The weights are fixed before official data access and have not been optimized on any test results.

In [ ]:
if len(WEIGHTS) != 2 or min(WEIGHTS) < 0 or not math.isclose(sum(WEIGHTS), 1.0):
    raise ValueError('Provide nonnegative weights summing to one.')
frozen_weights = tuple(WEIGHTS)

# 5. Evaluation

Enable once the input runs and weights are fixed. Every image uses the same scorer across all 333 candidates.

In [ ]:
RUN_FINAL_EVALUATION = False
result = None
if RUN_FINAL_EVALUATION:
    if C_RUN is None:
        raise ValueError('Set C_RUN to the completed 3Cv1 checkpoint folder.')
    c_ckpt = load_run(C_RUN, 'approach3c_dino_pairwise_feature_fusion', 'C',
                      dict(synthetic_pairs='observed_training_only', loss='moving_weighted_sum'))
    c_factory, _ = c_components(c_ckpt['hp'])
    c_model, c_meta = c_factory(c_ckpt['depth'], c_ckpt['metadata'])
    freeze(c_model, c_ckpt, c_meta)
    c_depth = c_ckpt['depth']
    batch_size = c_ckpt['hp']['batch_size']
    del c_ckpt

    if B_RUN is None:
        raise ValueError('Set B_RUN to the completed 3Bv2 checkpoint folder.')
    b_ckpt = load_run(B_RUN, 'approach3b_dino_text_alignment_v2', 'B', dict(pair=1., primitive=1.))
    b_factory, b_banks, b_predict_factory, _ = b_components(b_ckpt['hp'])
    b_model, text, b_meta = b_factory(pairs, concept_names, b_ckpt['depth'], b_ckpt['metadata'])
    freeze(b_model, b_ckpt, b_meta)
    b_depth = b_ckpt['depth']

    final_pairs = get_pairs(final_evaluation=True)
    candidates = final_pairs.to(cfg.DEVICE)
    crop_ids, disease_ids = candidates.unbind(1)

    banks, _ = b_banks(text, final_pairs, concept_names, b_ckpt['hp'])
    b_model.install_text(banks)
    b_predict = b_predict_factory(b_model, final_pairs)
    del b_ckpt, text

    @torch.no_grad()
    def predict(images):
        crop, disease = c_model.features(images.to(cfg.DEVICE))
        joint = c_model.score(crop, disease, candidates)
        return soft_vote((b_predict(images), joint), frozen_weights)

    hp = dict(weights=list(frozen_weights), b_depth=b_depth, c_depth=c_depth)

    seen, unseen = get_final_test_loaders(batch_size, norm='imagenet', final_evaluation=True)
    run = MODEL_NAME + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    previous = cfg.RESULTS_DIR
    RESULTS = previous / run
    RESULTS.mkdir(parents=True, exist_ok=False)
    try:
        cfg.RESULTS_DIR = RESULTS
        result = evaluate(predict, seen, unseen, MODEL_NAME, hp, robust=True)
    finally:
        cfg.RESULTS_DIR = previous

# 6. Save

The shared evaluator writes metrics and confusion files.

In [ ]:
if result is not None:
    print('Results:', RESULTS)